## KPI 1: Venta neta mensual

Observaciones:
- `variacion_pct` solo se calcula si el mes anterior existe y los dos meses están completos. Si no, es null
- Octubre no aparece (sin datos). Septiembre y noviembre son parciales

In [0]:
%sql

create or replace view fabriperezchauca.gold.kpi_venta_neta_mensual
comment 'KPI 1: venta neta por mes y variación contra el mes'
as

with por_mes as (
    select
        mes,
        first(dias_con_ventas_mes) as dias_con_ventas,
        first(estado_mes)          as estado_mes,
        sum(total)                 as venta_neta
    from fabriperezchauca.gold.ventas_obt
    group by mes
),
con_anterior as (
    select 
        *,
        lag(venta_neta) over(order by mes)                  as venta_neta_anterior,
        lag(estado_mes) over(order by mes)                  as estado_mes_anterior,
        lag(mes) over(order by mes) <=> add_months(mes, -1) as es_mes_consecutivo
    from por_mes
)

select 
    mes,
    dias_con_ventas,
    estado_mes,
    venta_neta,
    case
        when es_mes_consecutivo
            and estado_mes = 'completo'
            and estado_mes_anterior = 'completo'
            then round((venta_neta - venta_neta_anterior) / venta_neta_anterior * 100, 2)
    end as variacion_pct
from con_anterior



In [0]:
%sql
select * from fabriperezchauca.gold.kpi_venta_neta_mensual order by mes

## KPI 2: Top productos

Observaciones:
- 2a: ranking por unidades vendidas. 2b: ranking por ingresos. Ambos por mes
- Se excluyen `valor de ventas` (resumen, no es un producto) y productos null
- Ranking con `row_number()`: no repite posiciones. En empates se desempata por nombre de producto, para que el orden sea siempre el mismo

In [0]:
%sql

create or replace view fabriperezchauca.gold.kpi_top_productos
comment 'KPI 2: ranking mensual de productos por unidades (2a) y por ingresos (2b)'
as

with con_totales as (
    select 
        mes,
        estado_mes,
        producto,
        sum(cantidad) as unidades,
        sum(total)    as ingresos
    from fabriperezchauca.gold.ventas_obt
    where not es_resumen_ventas and producto is not null
    group by mes, estado_mes, producto
)

select
    *,
    row_number()  over(partition by mes order by unidades desc, producto asc) as rank_unidades,
    row_number()  over(partition by mes order by ingresos desc, producto asc) as rank_ingresos
from con_totales

In [0]:
%sql

-- Top ingresos

select 
    mes,
    estado_mes,
    producto,
    unidades,
    ingresos, 
    rank_ingresos
from fabriperezchauca.gold.kpi_top_productos
where mes = '2026-08-01' and rank_ingresos <= 5
order by rank_ingresos

In [0]:
%sql

-- Top unidades

select 
    mes,
    estado_mes,
    producto,
    unidades,
    ingresos, 
    rank_unidades 
from fabriperezchauca.gold.kpi_top_productos
where mes = '2026-08-01' and rank_unidades <= 5
order by rank_unidades

## KPI 3: Descuentos

Observaciones:
- Por mes: valor de lista, descuentos, % de descuento y recargos (contexto, no es KPI)
- Se excluyen las filas sin descuento calculado (falta algún precio o la cantidad), para que numerador y denominador usen las mismas filas
- Primero se suma, después se divide

In [0]:
%sql

create or replace view fabriperezchauca.gold.kpi_descuentos_mensual
comment 'KPI 3: descuentos por regateo por mes. Recargos como dato de contexto'
as

with por_mes as (
    select 
        mes,
        first(estado_mes)               as estado_mes,
        sum(cantidad * precio_original) as valor_lista,
        sum(descuento)                  as descuentos,
        sum(recargo)                    as recargos
    from fabriperezchauca.gold.ventas_obt
    where descuento is not null
    group by mes
)

select 
    *,
    round(descuentos / valor_lista * 100, 2) as pct_descuento
from por_mes

In [0]:
%sql
select * from fabriperezchauca.gold.kpi_descuentos_mensual order by mes

## KPI 4: Ingresos por forma de pago

Observaciones:
- Ingresos y participación % de cada forma de pago por mes
- Las ventas sin forma de pago se incluyen como `sin registrar`: si se excluyen, la suma no coincide con la venta neta del KPI 1

In [0]:
%sql

create or replace view fabriperezchauca.gold.kpi_formas_pago_mensual
comment 'KPI 4: ingresos y participación de cada forma de pago por mes'
as

with por_forma_pago as (
    select 
        mes,
        first(estado_mes)                     as estado_mes,
        coalesce(forma_pago, 'sin registrar') as forma_pago,
        sum(total)                            as ingresos
    from fabriperezchauca.gold.ventas_obt
    group by mes, coalesce(forma_pago, 'sin registrar')
)

select 
    *,
    round(ingresos / sum(ingresos) over(partition by mes) * 100, 2) as participacion_pct
from por_forma_pago

In [0]:
%sql

-- Cuadre de las participaciones, debe dar 99.99 o 100.01 por el redondeo

select 
    mes,
    sum(participacion_pct) as suma_pct
from fabriperezchauca.gold.kpi_formas_pago_mensual
group by mes
order by mes


In [0]:
%sql
select * 
from fabriperezchauca.gold.kpi_formas_pago_mensual
where mes = '2026-06-01'

## KPI 5: Ingresos por categoría

Observaciones:
- Ingresos y participación % de cada categoría por mes
- Las ventas sin categoría se incluyen como `sin registrar`, con el mismo criterio que el KPI 4

In [0]:
%sql

create or replace view fabriperezchauca.gold.kpi_categorias_mensual
comment 'KPI 4: ingresos y participación de cada categoria por mes'
as

with por_categoria as (
    select 
        mes,
        first(estado_mes)                     as estado_mes,
        coalesce(categoria, 'sin registrar')  as categoria,
        sum(total)                            as ingresos
    from fabriperezchauca.gold.ventas_obt
    group by mes, coalesce(categoria, 'sin registrar')
)

select 
    *,
    round(ingresos / sum(ingresos) over(partition by mes) * 100, 2) as participacion_pct
from por_categoria

In [0]:
%sql

-- Cuadre de las participaciones, debe dar 99.99 o 100.01 por el redondeo

select 
    mes,
    sum(participacion_pct) as suma_pct
from fabriperezchauca.gold.kpi_categorias_mensual
group by mes
order by mes


In [0]:
%sql
select * 
from fabriperezchauca.gold.kpi_categorias_mensual
where mes = '2026-08-01'

In [0]:
%sql

-- Validación final, deben cuadrar los montos en todos los kpi's

select 
    k1.mes,
    k1.venta_neta,
    k4.total_formas_pago,
    k5.total_categorias
from fabriperezchauca.gold.kpi_venta_neta_mensual k1
join 
    (select 
        mes,
        sum(ingresos) as total_formas_pago
    from fabriperezchauca.gold.kpi_formas_pago_mensual
    group by mes) k4 using (mes)
join
    (select 
        mes,
        sum(ingresos) as total_categorias
    from fabriperezchauca.gold.kpi_categorias_mensual
    group by mes) k5 using (mes)
order by mes

In [0]:
import requests

respuesta = requests.get("https://sheets.googleapis.com/$discovery/rest?version=v4", timeout=10)
print(respuesta.status_code)